# Sign in to Databricks Free Edition, query the Foundation Model API pay-per-token endpoint, and design a prompt that returns a strict JSON object

Your PM read a Hacker News thread about prompt engineering and now wants "a service that turns customer emails into clean JSON" shipped by Friday. The default GPT-style endpoint sometimes returns prose with a JSON object embedded somewhere in the middle, sometimes returns valid JSON with extra keys, and once returned a JSON object inside Markdown fenced code blocks. You have one session to design a prompt that returns strict JSON on every call, learn the temperature knob, and pick the right task category for the next three asks that are already queued behind this one.

The hands-on work:

- Open Databricks Free Edition, resolve the pay-per-token Foundation Model API endpoint `databricks-meta-llama-3-3-70b-instruct`, and call it via the OpenAI-compatible client.
- Author a system + user prompt pair that returns strict JSON with exactly three keys (`intent`, `priority`, `customer_id`) for three test customer-support emails the lab ships as a fixture.
- Sweep the temperature parameter from 0.0 to 1.0 on the same email, persist both responses, and see the deterministic-vs-creative trade-off in the row data.
- Pick the right Hugging Face / Foundation Model API task category for three queued business asks (a customer-review summarizer, a ticket-priority classifier, a marketing-copy drafter).

This is the GenAI Engineer Associate gateway lab. Every later lab assumes the Foundation Model API client works and the strict-JSON prompt pattern is in your fingers.

**Time.** Plan for about 50 minutes hands-on. Foundation Model API cold-start on the first call after idle can push you past 60 minutes. Budget up to 80 minutes for the session window.

**Cost.** About a penny. Foundation Model API pay-per-token bills against your workspace at fractions of a cent per call; roughly 30 calls across the lab adds up to about 1 cent of token spend. Free Edition swallows the compute and the metastore writes. A realistic session with re-runs and debugging lands at $0.01 to $0.03. Your morning coffee cost 200x more than this lab's token bill, even though the model has 70 billion parameters.

This lab maps to Databricks GenAI Engineer Associate Domain 1: Design Applications (~14% of exam weight, provisional).

In [ ]:
# NBVAL_SKIP
# Install labrun-checks, the Databricks SDK, and the OpenAI-compatible client.
# Pinned versions per LAB-CREATION-BLUEPRINT.md build rules. Never use unpinned
# installs.

!pip install --quiet labrun-checks==0.7.0 databricks-sdk==0.40.0 openai==1.54.0

In [ ]:
# NBVAL_SKIP
# Imports, per-lab constants, and the three-email fixture. UC identifiers
# (schema, tables) use snake_case under the labrun_ prefix because Unity
# Catalog identifiers prefer snake_case. The lab slug stays kebab-case for
# the tag value because slugs come from the cert YAML exactly as written.

import atexit
import getpass
import json
import re
import sys
import time

from databricks.sdk import WorkspaceClient
from databricks.sdk.service.sql import StatementState
from openai import OpenAI

from labrun_checks import (
    register,
    check,
    cleanup,
    run_cleanup,
    CheckpointResult,
    CleanupAdapter,
    CleanupResource,
)

LAB_ID = "prompt-engineering-and-foundation-model-api-basics"
LAB_TAG_KEY = "labrun_lab_slug"
LAB_TAG_VALUE = LAB_ID  # must equal cert YAML labs[0].slug exactly

CATALOG = "workspace"
PARENT_SCHEMA = "default"
LAB_SCHEMA = "labrun_prompt_basics"
LAB_SCHEMA_FQN = f"{CATALOG}.{PARENT_SCHEMA}.{LAB_SCHEMA}"
PROMPT_ITERATIONS_FQN = f"{LAB_SCHEMA_FQN}.prompt_iterations"
TEST_EMAILS_FQN = f"{LAB_SCHEMA_FQN}.test_emails"
TASK_PICKS_FQN = f"{LAB_SCHEMA_FQN}.task_picks"

# The Foundation Model API endpoint name. The lab also resolves the active
# endpoint dynamically via WorkspaceClient.serving_endpoints.list() so a
# quarterly model-rotation does not break the notebook.
FM_MODEL_HINT = "meta-llama-3"
FM_MODEL_DEFAULT = "databricks-meta-llama-3-3-70b-instruct"

# Three customer-support emails the lab uses for prompt engineering. One
# billing dispute, one shipping delay, one feature request. The bodies are
# deliberately messy (typos, mixed casing, run-on sentences) so the prompt
# has real work to do.
EMAILS = [
    {
        "email_id": 1,
        "body": (
            "hi team, my last invoice (cust id 84021) charged me twice for the "
            "june bill and i need this fixed asap, payroll is friday and i can "
            "not have $189 sitting in limbo on my card. i already called and "
            "the agent said i'd get a callback on tuesday but no one called. "
            "please escalate."
        ),
    },
    {
        "email_id": 2,
        "body": (
            "Order #84022 was supposed to arrive on Tuesday. UPS shows it stuck "
            "in Memphis since Saturday. I am leaving the country on Thursday "
            "morning. If this does not arrive before then can you reroute it "
            "to my office (customer 84022) or refund the shipping fee? this is "
            "the third late shipment in a row."
        ),
    },
    {
        "email_id": 3,
        "body": (
            "love the product. one ask: could you add a CSV export option to the "
            "reporting dashboard? our finance team copies numbers into excel "
            "every monday and the manual step adds about 40 minutes a week. "
            "customer 84023 here, happy to beta-test if useful. no rush, just "
            "flagging for the roadmap."
        ),
    },
]

In [ ]:
# NBVAL_SKIP
# Register the session, validate Databricks credentials via the SDK, resolve
# the Starter SQL warehouse, and stand up the OpenAI-compatible client
# against the workspace's serving endpoints base URL. This cell must succeed
# before the manifest cell runs anything, per LAB-CREATION-BLUEPRINT
# section 15.

session_token = getpass.getpass("Paste your session token from labrun.dev: ")
databricks_host = getpass.getpass(
    "Databricks workspace URL (https://...cloud.databricks.com or .azuredatabricks.net): "
).strip()
databricks_token = getpass.getpass("Databricks personal access token (PAT): ").strip()

if not databricks_host or not databricks_token:
    print("Workspace URL and PAT are both required. Re-run this cell.")
    raise SystemExit(1)
if not databricks_host.startswith("https://"):
    databricks_host = f"https://{databricks_host}"
databricks_host = databricks_host.rstrip("/")

w = WorkspaceClient(host=databricks_host, token=databricks_token)

try:
    me = w.current_user.me()
except Exception as exc:
    print("Databricks credentials are missing or expired. CurrentUser.me() failed:")
    print(f"  {exc}")
    print()
    print("Refresh your workspace URL and PAT, then re-run this cell.")
    raise SystemExit(1)

CALLER_USER = me.user_name or (me.emails[0].value if me.emails else "unknown")
print(f"Credentials validated. Workspace user: {CALLER_USER}")

warehouses = list(w.warehouses.list())
if not warehouses:
    print("No SQL warehouses found in this workspace. Free Edition ships with")
    print("a Starter warehouse by default; if it has been deleted, recreate it")
    print("from the SQL > Warehouses page before re-running this cell.")
    raise SystemExit(1)
WAREHOUSE = warehouses[0]
WAREHOUSE_ID = WAREHOUSE.id
print(f"SQL warehouse resolved: {WAREHOUSE.name} ({WAREHOUSE_ID})")

# Resolve the Foundation Model API endpoint dynamically. Endpoint names
# rotate every quarter; the lab matches on the substring meta-llama-3 and
# falls back to the documented default if no match is returned.
FM_ENDPOINT_NAME = FM_MODEL_DEFAULT
try:
    endpoints = list(w.serving_endpoints.list())
    matches = [
        ep.name for ep in endpoints
        if ep.name and FM_MODEL_HINT in ep.name and "instruct" in ep.name.lower()
    ]
    if matches:
        FM_ENDPOINT_NAME = matches[0]
except Exception as exc:
    print(f"Could not list serving endpoints (using default {FM_MODEL_DEFAULT}): {exc}")
print(f"Foundation Model API endpoint resolved: {FM_ENDPOINT_NAME}")

DATABRICKS_CREDENTIALS = {
    "host": databricks_host,
    "token": databricks_token,
    "warehouse_id": WAREHOUSE_ID,
    "fm_endpoint": FM_ENDPOINT_NAME,
}


def run_sql(statement, warehouse_id=None, wait_seconds=180):
    """Submit a SQL statement to the warehouse and return the parsed result.

    Polls statement state up to wait_seconds, raises on FAILED or CANCELED,
    returns a dict {columns: [...], rows: [[...]]} on SUCCEEDED.
    """
    wid = warehouse_id or WAREHOUSE_ID
    resp = w.statement_execution.execute_statement(
        warehouse_id=wid,
        statement=statement,
        wait_timeout="50s",
    )
    statement_id = resp.statement_id
    deadline = time.time() + wait_seconds
    while True:
        state = resp.status.state if resp.status else None
        if state == StatementState.SUCCEEDED:
            break
        if state in (StatementState.FAILED, StatementState.CANCELED, StatementState.CLOSED):
            err = resp.status.error.message if (resp.status and resp.status.error) else "no error message"
            raise RuntimeError(f"SQL failed ({state}): {err}\n  Statement: {statement}")
        if time.time() > deadline:
            raise TimeoutError(
                f"SQL did not finish in {wait_seconds}s. Last state: {state}. "
                f"The Starter warehouse may still be waking up; re-run the cell."
            )
        time.sleep(2)
        resp = w.statement_execution.get_statement(statement_id)

    columns = []
    if resp.manifest and resp.manifest.schema and resp.manifest.schema.columns:
        columns = [c.name for c in resp.manifest.schema.columns]
    rows = []
    if resp.result and resp.result.data_array:
        rows = list(resp.result.data_array)
    return {"columns": columns, "rows": rows}


# The OpenAI-compatible base URL points at the workspace's serving endpoints
# route. Same protocol shape as openai.com so the client library does not
# need a Databricks-specific build.
openai_client = OpenAI(
    api_key=databricks_token,
    base_url=f"{databricks_host}/serving-endpoints",
)
print(f"OpenAI-compatible client constructed against {databricks_host}/serving-endpoints")

register(session_token=session_token, lab_id=LAB_ID, credentials=DATABRICKS_CREDENTIALS)
print(f"Session registered for lab_id: {LAB_ID}")

In [ ]:
# NBVAL_SKIP
# Cleanup manifest, atexit safety net, and orphan scan.
# Manifest is module-level and in reverse-creation order. Free Edition has no
# hourly-billed resources for this lab (serverless compute auto-pauses,
# Foundation Model API pay-per-token is usage-billed, not hourly), so no
# entries are flagged critical. Per RESOURCE-SAFETY-SPEC Rule 4 the orphan
# scan blocks execution if any tagged resources from a prior session exist.


CLEANUP_MANIFEST = [
    CleanupResource(
        type="uc_managed_table",
        id=TASK_PICKS_FQN,
        region="databricks",
        cli_delete_command=f"databricks sql -e \"DROP TABLE IF EXISTS {TASK_PICKS_FQN}\"",
    ),
    CleanupResource(
        type="uc_managed_table",
        id=PROMPT_ITERATIONS_FQN,
        region="databricks",
        cli_delete_command=f"databricks sql -e \"DROP TABLE IF EXISTS {PROMPT_ITERATIONS_FQN}\"",
    ),
    CleanupResource(
        type="uc_managed_table",
        id=TEST_EMAILS_FQN,
        region="databricks",
        cli_delete_command=f"databricks sql -e \"DROP TABLE IF EXISTS {TEST_EMAILS_FQN}\"",
    ),
    CleanupResource(
        type="uc_schema",
        id=LAB_SCHEMA_FQN,
        region="databricks",
        cli_delete_command=f"databricks sql -e \"DROP SCHEMA IF EXISTS {LAB_SCHEMA_FQN} CASCADE\"",
    ),
]


class DatabricksCleanupAdapter:
    """Inline adapter implementing the labrun-checks CleanupAdapter protocol
    against Databricks Unity Catalog via the SQL Statement Execution API.
    Targets uc_managed_table and uc_schema for this lab."""

    def delete_resource(self, credentials, resource):
        rtype = resource.type
        rid = resource.id
        if rtype == "uc_managed_table":
            run_sql(f"DROP TABLE IF EXISTS {rid}")
        elif rtype == "uc_schema":
            run_sql(f"DROP SCHEMA IF EXISTS {rid} CASCADE")
        else:
            raise ValueError(f"DatabricksCleanupAdapter: unknown resource type {rtype!r}")

    def describe_resource(self, credentials, resource):
        rtype = resource.type
        rid = resource.id
        if rtype == "uc_managed_table":
            catalog, schema, table = rid.split(".")
            sql = (
                "SELECT 1 FROM system.information_schema.tables "
                f"WHERE table_catalog = '{catalog}' AND table_schema = '{schema}' "
                f"AND table_name = '{table}'"
            )
            result = run_sql(sql)
            return len(result["rows"]) > 0
        if rtype == "uc_schema":
            catalog, schema = rid.split(".")
            sql = (
                "SELECT 1 FROM system.information_schema.schemata "
                f"WHERE catalog_name = '{catalog}' AND schema_name = '{schema}'"
            )
            result = run_sql(sql)
            return len(result["rows"]) > 0
        return False

    def tag_scan(self, credentials, lab_slug, region):
        """Return list of UC identifiers tagged with the lab slug.

        Reads system.information_schema.schema_tags and
        system.information_schema.table_tags. Returns dotted FQNs so the
        identifiers match manifest ids."""
        arns = []
        queries = [
            (
                "SELECT catalog_name, schema_name FROM system.information_schema.schema_tags "
                f"WHERE tag_name = '{LAB_TAG_KEY}' AND tag_value = '{lab_slug}'",
                lambda row: f"{row[0]}.{row[1]}",
            ),
            (
                "SELECT catalog_name, schema_name, table_name FROM system.information_schema.table_tags "
                f"WHERE tag_name = '{LAB_TAG_KEY}' AND tag_value = '{lab_slug}'",
                lambda row: f"{row[0]}.{row[1]}.{row[2]}",
            ),
        ]
        for sql, fmt in queries:
            try:
                result = run_sql(sql)
            except Exception:
                continue
            for row in result["rows"]:
                arns.append(fmt(row))
        return arns


CLEANUP_ADAPTER = DatabricksCleanupAdapter()


def _atexit_cleanup():
    try:
        run_cleanup(CLEANUP_MANIFEST, adapter=CLEANUP_ADAPTER)
    except Exception:
        pass


atexit.register(_atexit_cleanup)


def scan_for_orphans():
    return CLEANUP_ADAPTER.tag_scan(DATABRICKS_CREDENTIALS, LAB_TAG_VALUE, "databricks")


orphans = scan_for_orphans()
if orphans:
    print(f"BLOCKED: existing UC objects tagged {LAB_TAG_KEY}={LAB_TAG_VALUE} were found:")
    for arn in orphans:
        print("  -", arn)
    print()
    print("These are leftovers from a previous run of this lab. Re-running")
    print("setup against an unclean state can produce duplicate resources.")
    print("Run the cleanup cell at the bottom of this notebook first, or")
    print("drop them manually with the SQL commands shown by the cleanup")
    print("cell, then re-run setup.")
    raise SystemExit(1)

print("No prior orphans found. Safe to create resources for this session.")

## Task 1: Reach the Foundation Model API endpoint with the OpenAI-compatible client

The Foundation Model API on Databricks speaks the OpenAI chat-completions protocol. Same `client.chat.completions.create(model=..., messages=...)` shape, same response object, same streaming semantics. The only Databricks-specific piece is the base URL: `<workspace_host>/serving-endpoints`. The PAT you pasted into setup is the `api_key`.

This task confirms three things at once:

1. Your PAT carries query access on pay-per-token endpoints (the Free Edition default).
2. The base URL is reachable and the endpoint name resolved correctly (Task 1 of the lab covers the workspace serving-endpoints surface, where the exam expects you to know the endpoint catalog is the routing layer).
3. The Llama 3.3 70B endpoint can serve a deterministic completion. The checkpoint asks the model for the single word `READY` at `temperature=0.0` and asserts the response contains it.

Use the `openai_client` object built in the setup cell and the `FM_ENDPOINT_NAME` constant. The endpoint name was resolved dynamically, so even if Databricks rotates the underlying model name next quarter, the variable already holds the right string.

In [ ]:
# NBVAL_SKIP
# Task 1: Issue a single deterministic chat completion against the
# Foundation Model API endpoint and capture the response text. The first
# call after idle can take 5 to 20 seconds (pay-per-token endpoints scale
# to zero on Free Edition).

print(f"Asking the Llama endpoint to warm up via {FM_ENDPOINT_NAME}...")

# YOUR CODE: Call openai_client.chat.completions.create with model=FM_ENDPOINT_NAME,
# temperature=0.0, and a single user message asking for the word READY.
# Assign the response to a variable named ready_response.

# Expect ready_response.choices[0].message.content to contain 'READY'.
response_text = ready_response.choices[0].message.content.strip()
print(f"Endpoint response: {response_text!r}")

In [ ]:
# NBVAL_SKIP
# Checkpoint 1: A fresh chat completion against the Foundation Model API
# endpoint returns a response containing the word READY. The validator
# builds its own OpenAI client (so a student who broke openai_client in
# Task 1 still gets a meaningful failure) and uses temperature=0.0 to keep
# the call deterministic.


def checkpoint_1(session):
    try:
        client = OpenAI(
            api_key=databricks_token,
            base_url=f"{databricks_host}/serving-endpoints",
        )
        completion = client.chat.completions.create(
            model=FM_ENDPOINT_NAME,
            messages=[
                {"role": "user", "content": "Reply with the single word READY."}
            ],
            temperature=0.0,
            max_tokens=16,
            timeout=60.0,
        )
        content = (completion.choices[0].message.content or "").strip().upper()
        if "READY" not in content:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Endpoint {FM_ENDPOINT_NAME} responded but the content did "
                    f"not contain 'READY'. Got: {content!r}. Re-issue the chat "
                    f"completion with model=FM_ENDPOINT_NAME and a user message "
                    f"asking for the single word READY."
                ),
            )
        return CheckpointResult(status="pass")
    except Exception as exc:
        return CheckpointResult(
            status="fail",
            error_reason=(
                f"Could not reach {FM_ENDPOINT_NAME} via the OpenAI-compatible "
                f"client: {exc}. Common causes: workspace URL missing https://, "
                f"PAT expired, or the endpoint name does not match what "
                f"WorkspaceClient.serving_endpoints.list() returns."
            ),
        )


check(1, checkpoint_1)

<details><summary>Hint 1 (nudge)</summary>

The checkpoint either timed out reaching the endpoint or the response came back without the word READY. The Foundation Model API endpoint is OpenAI-compatible, which means the same `client.chat.completions.create(...)` call shape works. Read the error message for which failure mode hit.

</details>

<details><summary>Hint 2 (direction)</summary>

One call: pass `model=FM_ENDPOINT_NAME`, `temperature=0.0`, and a single-element `messages` list with a user role asking the model to reply with the single word READY. Capture the return value as `ready_response`. The setup cell already built `openai_client` for you against `<databricks_host>/serving-endpoints`. The PAT you pasted is the `api_key`.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 1:

```python
ready_response = openai_client.chat.completions.create(
    model=FM_ENDPOINT_NAME,
    messages=[
        {"role": "user", "content": "Reply with the single word READY."}
    ],
    temperature=0.0,
    max_tokens=16,
)
```

</details>

**Wallet check.** About 0.05 cents of Foundation Model API spend. One round-trip through the Llama 3.3 70B endpoint at roughly 20 input + 5 output tokens. Pay-per-token billing on Free Edition is fractions of a cent per call; the running tally is still below 1 cent.

## Task 2: Design a strict-JSON prompt that holds for three different emails

The PM's ask is a contract: every email becomes a JSON object with exactly three keys (`intent`, `priority`, `customer_id`) and no Markdown fencing, no prose preamble, no extra keys. The exam-relevant work is the prompt grammar that produces that contract reliably, not the API call shape.

Build it in this order:

1. Create the lab schema under `workspace.default` and tag it so the orphan scan can find it: `CREATE SCHEMA IF NOT EXISTS workspace.default.labrun_prompt_basics` then `ALTER SCHEMA ... SET TAGS ('labrun_lab_slug' = 'prompt-engineering-and-foundation-model-api-basics')`.
2. Create two Delta tables in the schema and tag them: `test_emails` (the fixture, columns `email_id INT, body STRING`) and `prompt_iterations` (the working table, columns `id INT, email_id INT, system_prompt STRING, user_prompt STRING, temperature FLOAT, response_text STRING, parsed_ok BOOLEAN, recorded_at TIMESTAMP`). The cleanup cell drops both.
3. Insert the three fixture emails from the `EMAILS` constant into `test_emails`.
4. Author a `SYSTEM_PROMPT` and a `USER_PROMPT_TEMPLATE` such that calling the FM API at `temperature=0.0` on each email returns a string that parses via `json.loads()` and has exactly the keys `intent`, `priority`, `customer_id`.
5. For each of the three emails, call the FM API once, parse the response, and INSERT one row into `prompt_iterations` capturing the prompt pair, the response text, and the `parsed_ok` boolean (True if json.loads succeeded and the key set matched). Use `temperature=0.0` for these three rows; Task 3 adds the `temperature=1.0` rows.

The `customer_id` field shows up in every email as digits after the substring `customer` or `cust id`; the model should extract those four digits as an integer. The `intent` field should be one of `billing`, `shipping`, `feature_request`. The `priority` field should be one of `low`, `normal`, `high`, `urgent`.

The big risk: Llama 3.3 wraps JSON in Markdown fences (` ```json ... ``` `) intermittently, especially when the system prompt mentions JSON. Two defenses worth knowing for the exam:

- Pass `response_format={"type": "json_object"}` on the chat completion call. Many Databricks-hosted FM API endpoints support it; check the response.
- Post-process the response with a regex that strips Markdown fences before `json.loads`. Belt-and-suspenders for endpoints that ignore `response_format`.

In [ ]:
# NBVAL_SKIP
# Task 2: Stand up the lab schema and tables, design the strict-JSON
# prompts, run them against the three test emails at temperature=0.0, and
# persist the iterations. The CREATE SCHEMA / CREATE TABLE statements are
# idempotent so a kernel restart mid-lab does not blow up.

# YOUR CODE: Run CREATE SCHEMA IF NOT EXISTS for LAB_SCHEMA_FQN via run_sql(),
# then ALTER SCHEMA ... SET TAGS to apply LAB_TAG_KEY=LAB_TAG_VALUE.

# YOUR CODE: Run CREATE TABLE IF NOT EXISTS for TEST_EMAILS_FQN with
# (email_id INT, body STRING) USING DELTA, then ALTER TABLE ... SET TAGS.

# YOUR CODE: Run CREATE TABLE IF NOT EXISTS for PROMPT_ITERATIONS_FQN with
# columns (id INT, email_id INT, system_prompt STRING, user_prompt STRING,
# temperature FLOAT, response_text STRING, parsed_ok BOOLEAN,
# recorded_at TIMESTAMP) USING DELTA, then ALTER TABLE ... SET TAGS.

# Insert the three fixture emails. SQL string-escape the body so embedded
# apostrophes do not break the statement.
for email in EMAILS:
    body_escaped = email["body"].replace("'", "''")
    run_sql(
        f"INSERT INTO {TEST_EMAILS_FQN} VALUES "
        f"({email['email_id']}, '{body_escaped}')"
    )
print(f"Inserted {len(EMAILS)} fixture emails into {TEST_EMAILS_FQN}")


def strip_markdown_fences(text):
    """Defensive post-processor for endpoints that ignore response_format.
    Strips ```json and ``` fences plus surrounding whitespace."""
    cleaned = text.strip()
    cleaned = re.sub(r"^```(?:json)?\s*", "", cleaned, flags=re.IGNORECASE)
    cleaned = re.sub(r"\s*```\s*$", "", cleaned)
    return cleaned.strip()


REQUIRED_KEYS = {"intent", "priority", "customer_id"}

# YOUR CODE: Assign a SYSTEM_PROMPT string that instructs the model to
# return ONLY a JSON object with the three keys intent, priority,
# customer_id and forbids Markdown fences, prose, or extra keys.

# YOUR CODE: Assign a USER_PROMPT_TEMPLATE string with one {body} placeholder
# that will be .format()-interpolated with each email body.

# Run the prompt pair against each fixture email at temperature=0.0 and
# persist one row per email to prompt_iterations.
for row_id, email in enumerate(EMAILS, start=1):
    user_prompt = USER_PROMPT_TEMPLATE.format(body=email["body"])
    completion = openai_client.chat.completions.create(
        model=FM_ENDPOINT_NAME,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0.0,
        max_tokens=200,
        response_format={"type": "json_object"},
    )
    raw = completion.choices[0].message.content or ""
    cleaned = strip_markdown_fences(raw)
    parsed_ok = False
    try:
        parsed = json.loads(cleaned)
        parsed_ok = isinstance(parsed, dict) and set(parsed.keys()) == REQUIRED_KEYS
    except Exception:
        parsed_ok = False

    sys_escaped = SYSTEM_PROMPT.replace("'", "''")
    user_escaped = user_prompt.replace("'", "''")
    resp_escaped = cleaned.replace("'", "''")
    run_sql(
        f"INSERT INTO {PROMPT_ITERATIONS_FQN} VALUES ("
        f"{row_id}, {email['email_id']}, '{sys_escaped}', '{user_escaped}', "
        f"CAST(0.0 AS FLOAT), '{resp_escaped}', {str(parsed_ok).lower()}, "
        f"current_timestamp())"
    )
    print(f"email {email['email_id']} -> parsed_ok={parsed_ok}, response={cleaned[:80]!r}")

print(f"Persisted {len(EMAILS)} prompt iterations at temperature=0.0")

In [ ]:
# NBVAL_SKIP
# Checkpoint 2: All three test emails produced strict JSON with exactly
# the three required keys. Validator reads the persisted rows from
# prompt_iterations at temperature=0.0, re-parses each response_text via
# json.loads, and asserts the key set matches {intent, priority,
# customer_id} exactly for every email.


def checkpoint_2(session):
    try:
        sql = (
            "SELECT email_id, response_text, parsed_ok "
            f"FROM {PROMPT_ITERATIONS_FQN} "
            "WHERE temperature = CAST(0.0 AS FLOAT) "
            "ORDER BY email_id"
        )
        result = run_sql(sql)
        rows = result["rows"]
        if len(rows) < 3:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Expected at least 3 prompt_iterations rows at "
                    f"temperature=0.0; found {len(rows)}. Re-run Task 2 so the "
                    f"loop inserts one row per fixture email."
                ),
            )
        seen_email_ids = set()
        required = {"intent", "priority", "customer_id"}
        for row in rows:
            email_id = int(row[0])
            seen_email_ids.add(email_id)
            response_text = row[1] or ""
            try:
                parsed = json.loads(response_text)
            except Exception as exc:
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"Email {email_id}: response_text is not valid JSON "
                        f"after fence-stripping. json.loads raised {exc}. The "
                        f"response was: {response_text[:200]!r}. Use "
                        f"response_format={{'type': 'json_object'}} on the "
                        f"chat completion call OR strip Markdown fences "
                        f"before persisting."
                    ),
                )
            if not isinstance(parsed, dict):
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"Email {email_id}: JSON parsed but is not an object "
                        f"(got {type(parsed).__name__}). The system prompt must "
                        f"specify a JSON object, not an array or scalar."
                    ),
                )
            actual_keys = set(parsed.keys())
            if actual_keys != required:
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"Email {email_id}: JSON keys are {sorted(actual_keys)}; "
                        f"expected exactly {sorted(required)}. Tighten the "
                        f"system prompt to forbid extra keys and require all "
                        f"three."
                    ),
                )
        if {1, 2, 3} - seen_email_ids:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"prompt_iterations is missing rows for emails "
                    f"{sorted({1, 2, 3} - seen_email_ids)}. Re-run the Task 2 "
                    f"loop so every fixture email gets one temperature=0.0 row."
                ),
            )
        return CheckpointResult(status="pass")
    except Exception as exc:
        return CheckpointResult(status="error", error_reason=str(exc))


check(2, checkpoint_2)

<details><summary>Hint 1 (nudge)</summary>

The checkpoint error message names the failure mode: invalid JSON, wrong key set, or missing rows. If it is invalid JSON the model is almost certainly returning Markdown fences; if it is wrong key set the model is adding or dropping a key the system prompt did not pin down precisely; if it is missing rows the insert loop did not run for every email.

</details>

<details><summary>Hint 2 (direction)</summary>

For the schema and tables: three `CREATE` statements plus three `ALTER ... SET TAGS` statements, all wrapped in `run_sql()`. Constants `LAB_SCHEMA_FQN`, `TEST_EMAILS_FQN`, `PROMPT_ITERATIONS_FQN`, `LAB_TAG_KEY`, `LAB_TAG_VALUE` are in scope.

For the prompts: the system prompt must explicitly list the three required keys, give a one-line definition of each, forbid Markdown fences and extra keys, and pin the output to a single JSON object. The user prompt is one line with a `{body}` placeholder. Pass `response_format={"type": "json_object"}` on the chat completion call as the first line of defense; `strip_markdown_fences()` is the second.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 2:

```python
run_sql(f"CREATE SCHEMA IF NOT EXISTS {LAB_SCHEMA_FQN}")
run_sql(
    f"ALTER SCHEMA {LAB_SCHEMA_FQN} "
    f"SET TAGS ('{LAB_TAG_KEY}' = '{LAB_TAG_VALUE}')"
)

run_sql(
    f"CREATE TABLE IF NOT EXISTS {TEST_EMAILS_FQN} ("
    f"  email_id INT, body STRING"
    f") USING DELTA"
)
run_sql(
    f"ALTER TABLE {TEST_EMAILS_FQN} "
    f"SET TAGS ('{LAB_TAG_KEY}' = '{LAB_TAG_VALUE}')"
)

run_sql(
    f"CREATE TABLE IF NOT EXISTS {PROMPT_ITERATIONS_FQN} ("
    f"  id INT, email_id INT, system_prompt STRING, user_prompt STRING, "
    f"  temperature FLOAT, response_text STRING, parsed_ok BOOLEAN, "
    f"  recorded_at TIMESTAMP"
    f") USING DELTA"
)
run_sql(
    f"ALTER TABLE {PROMPT_ITERATIONS_FQN} "
    f"SET TAGS ('{LAB_TAG_KEY}' = '{LAB_TAG_VALUE}')"
)

SYSTEM_PROMPT = (
    "You parse customer-support emails into a JSON object. "
    "Return ONLY a single JSON object. No Markdown, no fenced code blocks, "
    "no prose, no extra keys. "
    "The object MUST have exactly these three keys: "
    "'intent' (one of: 'billing', 'shipping', 'feature_request'), "
    "'priority' (one of: 'low', 'normal', 'high', 'urgent'), "
    "'customer_id' (integer; extract the digits following 'customer', "
    "'cust id', or 'order #' in the email body). "
    "If a field is not present in the email, infer the most plausible value "
    "from the body but never omit the key."
)

USER_PROMPT_TEMPLATE = (
    "Email body:\n{body}\n\n"
    "Return the JSON object now."
)
```

</details>

**Wallet check.** About 0.5 cents of Foundation Model API spend so far. Three chat completions at roughly 250 tokens each (system prompt + user prompt + JSON response). Running tally: still under 1 cent. Your coffee is still winning by 400x.

## Task 3: Sweep the temperature parameter and prove the outputs diverge

The exam's Domain 1 task statement on prompt design names temperature as one of the knobs the engineer tunes. The intuition pump: at `temperature=0.0` the decoder picks the single highest-probability next token at every step, so the output is byte-identical across reruns (assuming the same prompt and the same model weights). At `temperature=1.0` the decoder samples proportionally from the probability distribution, so the output varies run to run.

For a strict-JSON contract this is usually a footgun (T>0 increases the chance of hallucinated keys, hallucinated values, or fence-wrapping). But there are use cases where non-zero temperature is the right call: marketing copy that should not sound robotic, brainstorming where you want N different angles, creative writing benchmarks. The lab proves the divergence so the trade-off is in your hands.

Build it:

1. Pick `email_id = 1` (the billing email).
2. Run the same `SYSTEM_PROMPT` + `USER_PROMPT_TEMPLATE.format(body=...)` pair through the FM API at `temperature=1.0`. To raise the chance of divergence, call it twice (each call is independent at T=1.0; the two responses may differ from each other and from the T=0.0 response).
3. Persist both temperature=1.0 responses to `prompt_iterations` with `temperature = 1.0`.
4. The checkpoint runs `SELECT COUNT(DISTINCT response_text) FROM prompt_iterations WHERE email_id = 1 AND temperature IN (0.0, 1.0)` and asserts the distinct count is at least 2. Distinct outputs prove the temperature knob does what the exam expects you to know it does.

In [ ]:
# NBVAL_SKIP
# Task 3: Run the same prompt against email 1 at temperature=1.0 twice and
# persist both responses to prompt_iterations.

TARGET_EMAIL = next(e for e in EMAILS if e["email_id"] == 1)
user_prompt_t1 = USER_PROMPT_TEMPLATE.format(body=TARGET_EMAIL["body"])

# Find the next id to use (the prompt_iterations table already has three
# rows from Task 2). Re-running this cell is safe; we keep appending.
count_result = run_sql(f"SELECT MAX(id) FROM {PROMPT_ITERATIONS_FQN}")
base_id = int(count_result["rows"][0][0] or 0)

for offset in (1, 2):
    # YOUR CODE: Call openai_client.chat.completions.create with model=FM_ENDPOINT_NAME,
    # the same SYSTEM_PROMPT and user_prompt_t1, temperature=1.0,
    # max_tokens=200, and response_format={'type': 'json_object'}. Assign the
    # return value to a variable named completion.

    raw = completion.choices[0].message.content or ""
    cleaned = strip_markdown_fences(raw)
    parsed_ok = False
    try:
        parsed = json.loads(cleaned)
        parsed_ok = isinstance(parsed, dict) and set(parsed.keys()) == REQUIRED_KEYS
    except Exception:
        parsed_ok = False

    new_id = base_id + offset
    sys_escaped = SYSTEM_PROMPT.replace("'", "''")
    user_escaped = user_prompt_t1.replace("'", "''")
    resp_escaped = cleaned.replace("'", "''")
    run_sql(
        f"INSERT INTO {PROMPT_ITERATIONS_FQN} VALUES ("
        f"{new_id}, {TARGET_EMAIL['email_id']}, '{sys_escaped}', '{user_escaped}', "
        f"CAST(1.0 AS FLOAT), '{resp_escaped}', {str(parsed_ok).lower()}, "
        f"current_timestamp())"
    )
    print(f"temperature=1.0 call {offset}: parsed_ok={parsed_ok}, response={cleaned[:80]!r}")

distinct = run_sql(
    f"SELECT COUNT(DISTINCT response_text) FROM {PROMPT_ITERATIONS_FQN} "
    f"WHERE email_id = 1 AND temperature IN (CAST(0.0 AS FLOAT), CAST(1.0 AS FLOAT))"
)
print(f"Distinct response_text values for email 1 across T=0.0 and T=1.0: {distinct['rows'][0][0]}")

In [ ]:
# NBVAL_SKIP
# Checkpoint 3: prompt_iterations holds at least one row at temperature=0.0
# AND at least one row at temperature=1.0 for email_id=1, AND the distinct
# count of response_text across those rows is at least 2. Distinct outputs
# at different temperatures prove the parameter is doing real work.


def checkpoint_3(session):
    try:
        cov_sql = (
            "SELECT "
            "  SUM(CASE WHEN temperature = CAST(0.0 AS FLOAT) THEN 1 ELSE 0 END) AS t0_rows, "
            "  SUM(CASE WHEN temperature = CAST(1.0 AS FLOAT) THEN 1 ELSE 0 END) AS t1_rows "
            f"FROM {PROMPT_ITERATIONS_FQN} WHERE email_id = 1"
        )
        cov_result = run_sql(cov_sql)
        if not cov_result["rows"]:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"No rows in {PROMPT_ITERATIONS_FQN} for email_id=1. "
                    f"Re-run Task 2 then Task 3 so the table holds the "
                    f"T=0.0 row and at least one T=1.0 row."
                ),
            )
        t0_rows = int(cov_result["rows"][0][0] or 0)
        t1_rows = int(cov_result["rows"][0][1] or 0)
        if t0_rows < 1:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"No temperature=0.0 row for email_id=1. Task 2 should "
                    f"have written it; re-run Task 2 first."
                ),
            )
        if t1_rows < 1:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"No temperature=1.0 row for email_id=1. Task 3 must "
                    f"INSERT at least one row at temperature=1.0."
                ),
            )
        distinct_sql = (
            "SELECT COUNT(DISTINCT response_text) "
            f"FROM {PROMPT_ITERATIONS_FQN} WHERE email_id = 1 "
            "AND temperature IN (CAST(0.0 AS FLOAT), CAST(1.0 AS FLOAT))"
        )
        distinct_result = run_sql(distinct_sql)
        distinct_count = int(distinct_result["rows"][0][0] or 0)
        if distinct_count < 2:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Distinct response_text count across T=0.0 and T=1.0 for "
                    f"email_id=1 is {distinct_count}; expected at least 2. "
                    f"Llama 3.3 70B can produce byte-identical output at "
                    f"T=1.0 occasionally; re-run Task 3 once more so a fresh "
                    f"sample lands in the table."
                ),
            )
        return CheckpointResult(status="pass")
    except Exception as exc:
        return CheckpointResult(status="error", error_reason=str(exc))


check(3, checkpoint_3)

<details><summary>Hint 1 (nudge)</summary>

The checkpoint either says no T=1.0 row exists (the call did not write to the table) or says the distinct count is 1 (T=1.0 happened to produce the same bytes as T=0.0 on this run). The error message tells you which. Re-running this task once more usually shakes out the second failure mode because T=1.0 sampling is stochastic across calls.

</details>

<details><summary>Hint 2 (direction)</summary>

Two calls in the loop, both at `temperature=1.0`, both with the same `SYSTEM_PROMPT` and `user_prompt_t1`. Pass `max_tokens=200` and `response_format={"type": "json_object"}` exactly like Task 2 did. Capture each return value as `completion`; the rest of the loop body handles parsing, fence-stripping, and the INSERT.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 3 (the body of the loop):

```python
completion = openai_client.chat.completions.create(
    model=FM_ENDPOINT_NAME,
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt_t1},
    ],
    temperature=1.0,
    max_tokens=200,
    response_format={"type": "json_object"},
)
```

</details>

**Wallet check.** About 1 cent of Foundation Model API spend so far. Five total chat completions across the lab; the temperature sweep was the heaviest task because it issued two extra calls. Running tally: roughly 1 cent. Your coffee remains 200x more expensive.

## Task 4: Pick the right task category for three queued business asks

Domain 1 of the exam includes the task statement "Select model tasks (summarization, classification, extraction, text2text generation, NER, sentencizer) to accomplish a given business requirement." The exam tests vocabulary: the engineer who can name the Hugging Face / Foundation Model API task category for a given business ask picks the right pre-evaluated model pool, the right benchmark metrics, and the right downstream chain shape.

Three asks landed in the queue behind the strict-JSON parser. Read each ask, pick the canonical task category, and persist your picks to a new `task_picks` table so the checkpoint can read them back.

**Ask 1: customer-review summarizer.** The product team has a Delta table of 500 customer reviews (200 to 800 words each). They want a one-sentence executive summary per review for the on-call lead's phone notification. Which task category?

**Ask 2: ticket-priority classifier.** The support team wants every incoming ticket auto-labeled as `low`, `normal`, `high`, or `urgent` based on the ticket body alone. The output is a single label drawn from a fixed set of four. Which task category?

**Ask 3: marketing-copy drafter.** The growth team wants three to five draft social posts per new product feature, in different tones (casual, formal, playful). Each draft is freeform prose, no fixed schema. Which task category?

Build it:

1. Create the `task_picks` table with columns `ask_id INT, picked_task STRING` (USING DELTA, tag it with the lab tag).
2. INSERT three rows with your picks for ask 1, ask 2, ask 3.
3. The checkpoint asserts (ask 1 = `summarization`, ask 2 = `text-classification`, ask 3 = `text-generation`). Lowercase, hyphenated where the canonical category uses a hyphen. Match the Hugging Face Tasks page conventions, not the prose phrasing from the ask.

In [ ]:
# NBVAL_SKIP
# Task 4: Create the task_picks table and persist three rows with the
# picked task category per ask.

# YOUR CODE: Run CREATE TABLE IF NOT EXISTS for TASK_PICKS_FQN with columns
# (ask_id INT, picked_task STRING) USING DELTA via run_sql(), then ALTER
# TABLE ... SET TAGS to apply LAB_TAG_KEY=LAB_TAG_VALUE.

# YOUR CODE: INSERT three rows into TASK_PICKS_FQN with your picked task
# category per ask. Use lowercase, hyphenated where canonical (e.g.,
# 'text-classification', not 'classification').

result = run_sql(
    f"SELECT ask_id, picked_task FROM {TASK_PICKS_FQN} ORDER BY ask_id"
)
for row in result["rows"]:
    print(f"ask {row[0]}: {row[1]}")

In [ ]:
# NBVAL_SKIP
# Checkpoint 4: task_picks has exactly three rows, one per ask_id in
# {1, 2, 3}, with picked_task matching the expected canonical task
# categories.


def checkpoint_4(session):
    try:
        sql = (
            "SELECT ask_id, picked_task "
            f"FROM {TASK_PICKS_FQN} ORDER BY ask_id"
        )
        result = run_sql(sql)
        rows = result["rows"]
        if len(rows) < 3:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"task_picks has {len(rows)} rows; expected exactly 3, "
                    f"one per ask_id in (1, 2, 3). Re-run the INSERT."
                ),
            )
        expected = {
            1: "summarization",
            2: "text-classification",
            3: "text-generation",
        }
        actual = {}
        for row in rows:
            ask_id = int(row[0])
            picked = (row[1] or "").strip().lower()
            actual[ask_id] = picked
        mismatches = []
        for ask_id, want in expected.items():
            got = actual.get(ask_id)
            if got != want:
                mismatches.append(f"ask {ask_id}: expected {want!r}, got {got!r}")
        if mismatches:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    "Task-category picks do not match the expected canonical "
                    "set. " + "; ".join(mismatches) + ". The canonical strings "
                    "are 'summarization' (ask 1), 'text-classification' "
                    "(ask 2), 'text-generation' (ask 3). Lowercase, hyphenated "
                    "where Hugging Face hyphenates."
                ),
            )
        return CheckpointResult(status="pass")
    except Exception as exc:
        return CheckpointResult(status="error", error_reason=str(exc))


check(4, checkpoint_4)

<details><summary>Hint 1 (nudge)</summary>

The checkpoint compares your three rows against an exact canonical set. The error message tells you which ask_id has the wrong pick. Check the Hugging Face Tasks page (huggingface.co/tasks) for the canonical category names; a wrong pick here almost always means choosing the umbrella category over the specific one.

</details>

<details><summary>Hint 2 (direction)</summary>

For ask 1 the output is shorter than the input: pick the category that means "produce a shorter version of a longer text." For ask 2 the output is a single label drawn from a fixed set: pick the category that means "assign a class label." For ask 3 the output is open-ended prose: pick the umbrella category. Lowercase, hyphenated where the canonical name hyphenates. The strings the checkpoint accepts are: `summarization`, `text-classification`, `text-generation`.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 4:

```python
run_sql(
    f"CREATE TABLE IF NOT EXISTS {TASK_PICKS_FQN} ("
    f"  ask_id INT, picked_task STRING"
    f") USING DELTA"
)
run_sql(
    f"ALTER TABLE {TASK_PICKS_FQN} "
    f"SET TAGS ('{LAB_TAG_KEY}' = '{LAB_TAG_VALUE}')"
)
run_sql(
    f"INSERT INTO {TASK_PICKS_FQN} VALUES "
    f"(1, 'summarization'), "
    f"(2, 'text-classification'), "
    f"(3, 'text-generation')"
)
```

If you need to reset because you inserted the wrong values:

```python
run_sql(f"TRUNCATE TABLE {TASK_PICKS_FQN}")
run_sql(
    f"INSERT INTO {TASK_PICKS_FQN} VALUES "
    f"(1, 'summarization'), "
    f"(2, 'text-classification'), "
    f"(3, 'text-generation')"
)
```

</details>

**Wallet check.** Still about 1 cent. Task 4 was three INSERTs and a SELECT against the warehouse, no Foundation Model API calls. The token tally has not moved since Task 3.

## Cleanup

Time to tear it all down. The cell below runs through your manifest in reverse-creation order (task_picks, prompt_iterations, test_emails, then the schema with `CASCADE`), then double-checks UC information_schema to confirm everything is gone. Re-running is safe.

In [ ]:
# NBVAL_SKIP
# Cleanup. Tear down every resource in CLEANUP_MANIFEST in reverse-creation
# order using the inline Databricks adapter. No critical (hourly-billed)
# resources in this lab, so the canonical summary always reports zero
# critical destructions.

result = run_cleanup(CLEANUP_MANIFEST, adapter=CLEANUP_ADAPTER)

for warning in result.warnings:
    print(warning)
for orphan in result.orphans:
    print(orphan)
if result.warnings or result.orphans:
    print()

failed_ids = set()
for warning_msg in result.warnings:
    for res in CLEANUP_MANIFEST:
        if res.id in warning_msg:
            failed_ids.add(res.id)
            break

critical_total = 0
standard_total = len(CLEANUP_MANIFEST)
critical_destroyed = critical_total
standard_destroyed = standard_total - len(failed_ids)
failed_count = len(failed_ids)

print("Cleanup complete.")
print(f"Critical resources destroyed: {critical_destroyed}")
print(f"Standard resources destroyed: {standard_destroyed}")
print(f"Resources that failed to delete: {failed_count} (see above for CLI commands)")
print("If K > 0, your Databricks workspace may still hold lab objects. Resolve before closing this notebook.")

cleanup(status=result.status)

if failed_count > 0:
    sys.exit(1)

**Session total: about 1 cent of Foundation Model API spend.** Roughly 8 to 12 chat completions across the four tasks at ~250 tokens each lands in the $0.01 to $0.03 band for a clean run. Free Edition swallowed the compute, the metastore writes, and the warehouse-seconds. The three Delta tables and the lab schema were destroyed by the cleanup cell above, so the daily token quota is the only thing this session touched, and it resets at 00:00 UTC. Your morning coffee cost 200x more than this lab's GPU bill, even though the model has 70 billion parameters and there is no GPU on your laptop.

## Reflection

These are not graded. They are for you.

1. Walk through what the Foundation Model API serving endpoint does when you POST a chat completion request to it. Name each layer the request traverses (workspace authentication, AI Gateway routing if enabled, model container, tokenizer, model forward pass, response stream). What does Unity Catalog enforce, if anything, at each layer?

2. Your prompt at `temperature=0.0` returned the same byte-identical JSON across three reruns; at `temperature=1.0` the JSON keys were the same but the `intent` field value varied. Why does temperature matter for a structured-output use case, and when would you intentionally choose a non-zero temperature?

## Exam-Style Practice

**Question 1 (Domain 1, task category selection per the official sample-question pattern):**

A GenAI engineer is building an application that takes an internal helpdesk ticket (200 to 800 words) and produces a one-sentence executive summary the on-call lead reads on a phone notification. With which NLP task category should they evaluate potential LLMs?

A. text-generation, because every chat completion is text generation underneath.

B. Summarization, because the task category is "produce a shorter version of a longer input."

C. Text classification, because the output is a single label per input.

D. Sentencizer, because the output is exactly one sentence.

<details><summary>Show answer</summary>

**Correct: B.**

- A is wrong: text-generation is the general open-ended category; choosing it skips the more specific Hugging Face task category that the exam expects you to name. The exam's task-category pattern (Question 5 in the official sample-question set is the same shape) rewards the specific category over the umbrella one.
- B is correct: Summarization is the Hugging Face / Foundation Model API task category for "shorten a longer input to its gist." Model cards under the Summarization task tag are pre-evaluated on ROUGE; that is the relevant evaluator pool.
- C is wrong: Text classification returns a label from a fixed set (positive, neutral, negative, etc.). The output here is freeform sentence text, not a label.
- D is wrong: Sentencizer is a low-level text-processing task (splitting a paragraph into sentence boundaries); it does not generate new content.

</details>

**Question 2 (Domain 1, strict-JSON prompt design):**

A GenAI engineer needs an LLM to return a JSON object with exactly three keys (`intent`, `priority`, `customer_id`) and no Markdown fencing. Their first prompt returns valid JSON wrapped in a ` ```json ... ``` ` fenced code block. Which approach is the most reliable fix?

A. Lower the temperature to 0.0 and rely on the deterministic decoding to never produce Markdown fences.

B. Use the `response_format={"type": "json_object"}` parameter on the chat completion call if the endpoint supports it; otherwise post-process the response with a regex that strips Markdown fences before parsing.

C. Switch to a smaller model (the smaller model will not have been trained on Markdown-heavy data).

D. Ask the model in the user prompt to "please do not use Markdown" and trust the instruction.

<details><summary>Show answer</summary>

**Correct: B.**

- A is wrong: temperature=0.0 makes the output deterministic but does NOT prevent the model from producing Markdown fences if its training data leaned toward fenced JSON for this type of prompt. The model is deterministically wrong if the prompt happens to elicit fences.
- B is correct: the `response_format={"type": "json_object"}` parameter (OpenAI-compatible feature also supported by many Databricks Foundation Model API endpoints) constrains the model's decoder to produce valid JSON. The defensive post-processing is the belt-and-suspenders fallback for endpoints that do not support the parameter.
- C is wrong: smaller models do not have a smaller fence-producing bias; this trade off would also degrade response quality on harder asks.
- D is wrong: politely asking the model is the lowest-reliability approach. Llama and similar models intermittently ignore instructions, especially at non-zero temperature. The exam's bias is toward structural fixes (parameter, post-processing) over prompt-text fixes for hard structural constraints.

</details>